# Unit stability

This notebook checks whether units that pass the standard quality criteria have stable spike amplitudes and flags non-unimodal amplitude distributions as an additional control.

Amplitude drift is measured across equal time windows. Hartigan's dip test is applied once per unit, and its p-values are corrected across units within each probe and session. Units with more than 72,000 spikes use a fixed random sample of 72,000 amplitudes because the dip-test package's tabulated p-values stop there.

In [ ]:
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from diptest import diptest
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from statsmodels.stats.multitest import multipletests

# Silence the setuptools pkg_resources deprecation notice
warnings.filterwarnings("ignore", category=UserWarning, module="datajoint.plugin")

from labdata.schema import EphysRecording, SpikeSorting, UnitCount  # noqa: E402

%matplotlib widget

Set the subject, sessions, quality criteria, and stability thresholds.

In [ ]:
subject_name = "GRB058"
session_names = ["20260224_152424"]  # Set to None to analyze all sorted sessions.

liberal_criteria_id = 0
standard_criteria_id = 1
amp_drift_threshold = 0.10
dip_alpha = 0.05
max_dip_samples = 72_000

n_chunks = 5
n_bins = 50

Define the per-unit amplitude checks and correct dip-test p-values within each probe and session.

In [ ]:
def analyze_unit(
    spike_times, spike_amplitudes, recording_end, n_chunks=5, max_dip_samples=72_000
):
    """Measure amplitude drift over equal time windows and test unimodality."""
    spike_times = np.asarray(spike_times)
    spike_amplitudes = np.asarray(spike_amplitudes)
    if len(spike_times) != len(spike_amplitudes):
        raise ValueError("Spike times and amplitudes must have the same length.")

    chunk_index = np.digitize(
        spike_times, np.linspace(0, recording_end, n_chunks + 1)[1:-1]
    )
    chunks = [spike_amplitudes[chunk_index == i] for i in range(n_chunks)]
    chunk_means = np.array([chunk.mean() if len(chunk) else np.nan for chunk in chunks])
    mean_amplitude = spike_amplitudes.mean()
    drift = (
        (np.nanmax(chunk_means) - np.nanmin(chunk_means)) / mean_amplitude
        if mean_amplitude
        else np.nan
    )
    dip_sample = spike_amplitudes
    if len(dip_sample) > max_dip_samples:
        dip_sample = np.random.default_rng(0).choice(
            dip_sample, max_dip_samples, replace=False
        )
    dip, dip_p = diptest(dip_sample)
    return {
        "amplitude_drift": drift,
        "dip": dip,
        "dip_p": dip_p,
        "dip_n": len(dip_sample),
        "chunks": chunks,
    }


def analyze_probe_stability(
    units_df,
    liberal_unit_count,
    recording_end,
    amp_drift_threshold=0.10,
    dip_alpha=0.05,
    n_chunks=5,
    max_dip_samples=72_000,
):
    analysis_df = pd.DataFrame(
        [
            {
                "unit_id": row.unit_id,
                **analyze_unit(
                    row.spike_times,
                    row.spike_amplitudes,
                    recording_end,
                    n_chunks,
                    max_dip_samples,
                ),
            }
            for row in units_df.itertuples()
        ]
    )
    _, analysis_df["dip_q"], _, _ = multipletests(
        analysis_df["dip_p"], alpha=dip_alpha, method="fdr_bh"
    )
    analysis_df["non_unimodal"] = analysis_df["dip_q"] < dip_alpha
    analysis_df["high_amplitude_drift"] = (
        analysis_df["amplitude_drift"] > amp_drift_threshold
    )

    counts = [
        liberal_unit_count,
        len(analysis_df),
        int((~analysis_df["high_amplitude_drift"]).sum()),
    ]
    return analysis_df, counts

Fetch units that already pass labdata criteria 0 and 1, then run the stability checks for each probe.

In [ ]:
if session_names is None:
    session_names = sorted(
        set((SpikeSorting & {"subject_name": subject_name}).fetch("session_name"))
    )

change_in_unit_counts = {}
probe_widget_data = {}
unit_key_fields = [
    "subject_name",
    "session_name",
    "dataset_name",
    "probe_num",
    "parameter_set_num",
    "unit_id",
]

for session_name in session_names:
    sorting_keys = (
        SpikeSorting & {"subject_name": subject_name, "session_name": session_name}
    ).fetch("KEY")

    for sorting_key in sorting_keys:
        liberal_units = (
            UnitCount.Unit
            & sorting_key
            & {"unit_criteria_id": liberal_criteria_id, "passes": 1}
        )
        standard_units = (
            UnitCount.Unit
            & sorting_key
            & {"unit_criteria_id": standard_criteria_id, "passes": 1}
        )
        if not len(liberal_units) or not len(standard_units):
            raise RuntimeError(
                f"UnitCount is missing for {session_name}, probe "
                f"{sorting_key['probe_num']}. Run unit_count_setup first."
            )

        unit_keys = standard_units.fetch(*unit_key_fields, as_dict=True)
        units_df = pd.DataFrame(
            (SpikeSorting.Unit & unit_keys).fetch(
                "unit_id", "spike_times", "spike_amplitudes", as_dict=True
            )
        ).sort_values("unit_id")

        recording_duration, sampling_rate = (
            EphysRecording * EphysRecording.ProbeSetting & sorting_key
        ).fetch1("recording_duration", "sampling_rate")
        recording_end = float(recording_duration) * float(sampling_rate)

        analysis_df, counts = analyze_probe_stability(
            units_df,
            liberal_unit_count=len(liberal_units),
            recording_end=recording_end,
            amp_drift_threshold=amp_drift_threshold,
            dip_alpha=dip_alpha,
            n_chunks=n_chunks,
            max_dip_samples=max_dip_samples,
        )

        probe_num = sorting_key["probe_num"]
        change_in_unit_counts.setdefault(probe_num, []).append(counts)
        probe_widget_data[(session_name, probe_num)] = analysis_df
        print(
            f"{session_name} imec{probe_num}: {len(analysis_df)} standard units, "
            f"{analysis_df['high_amplitude_drift'].sum()} high-drift, "
            f"{analysis_df['non_unimodal'].sum()} non-unimodal"
        )

Plot how the standard depth criterion and the amplitude-stability check change unit counts.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = [
    "Liberal quality\n(criteria 0)",
    "Standard quality\n(criteria 1)",
    "Standard + stable\namplitude",
]

for probe_num, counts_list in sorted(change_in_unit_counts.items()):
    counts = np.asarray(counts_list)
    for session_counts in counts:
        ax.plot(labels, session_counts, color="tab:blue", alpha=0.25)
    ax.plot(
        labels,
        counts.mean(axis=0),
        marker="o",
        color="tab:blue",
        label=f"imec{probe_num} mean",
    )
    total_change = 100 * (counts[:, 0].sum() - counts[:, -1].sum()) / counts[:, 0].sum()
    print(f"imec{probe_num}: total reduction = {total_change:.1f}%")

ax.set_title("Unit counts after quality and amplitude-stability checks")
ax.set_ylabel("Unit count")
ax.legend()
fig.tight_layout()

Define a browser for viewing equal-time amplitude histograms and each unit's two control flags.

In [ ]:
class StabilityBrowser:
    FILTER_OPTIONS = [
        "All",
        "High amplitude drift",
        "Non-unimodal",
        "Both flags",
        "Stable amplitude",
    ]

    def __init__(self, data, amp_drift_threshold=0.10, dip_alpha=0.05, n_bins=50):
        self.data = data
        self.amp_drift_threshold = amp_drift_threshold
        self.dip_alpha = dip_alpha
        self.n_bins = n_bins
        self._fig: Figure | None = None
        self._ax: Axes | None = None
        self._updating = False

        self.session_dropdown = widgets.Dropdown(
            options=sorted({key[0] for key in data}), description="Session:"
        )
        self.probe_dropdown = widgets.Dropdown(description="Probe:")
        self.filter_radio = widgets.RadioButtons(
            options=self.FILTER_OPTIONS, description="Show:"
        )
        self.unit_slider = widgets.SelectionSlider(
            options=["—"],
            description="Unit:",
            continuous_update=False,
            layout=widgets.Layout(width="420px"),
        )
        self.count_label = widgets.HTML()
        self.stats_html = widgets.HTML()

        self.session_dropdown.observe(self._on_session_change, names="value")
        self.probe_dropdown.observe(self._refresh_units, names="value")
        self.filter_radio.observe(self._refresh_units, names="value")
        self.unit_slider.observe(self._update_display, names="value")
        self._on_session_change()

    def _key(self):
        return (
            self.session_dropdown.value,
            int(self.probe_dropdown.value.removeprefix("imec")),
        )

    def _on_session_change(self, _change=None):
        self._updating = True
        probes = sorted(
            key[1] for key in self.data if key[0] == self.session_dropdown.value
        )
        self.probe_dropdown.options = [f"imec{probe}" for probe in probes]
        self.probe_dropdown.value = f"imec{probes[0]}"
        self._updating = False
        self._refresh_units()

    def _filtered(self):
        df = self.data[self._key()]
        selected = self.filter_radio.value
        if selected == "High amplitude drift":
            return df[df["high_amplitude_drift"]]
        if selected == "Non-unimodal":
            return df[df["non_unimodal"]]
        if selected == "Both flags":
            return df[df["high_amplitude_drift"] & df["non_unimodal"]]
        if selected == "Stable amplitude":
            return df[~df["high_amplitude_drift"]]
        return df

    def _refresh_units(self, _change=None):
        if self._updating or not self.probe_dropdown.value:
            return
        self._updating = True
        filtered = self._filtered()
        options = filtered["unit_id"].tolist() or ["—"]
        self.unit_slider.options = options
        self.unit_slider.value = options[0]
        self.count_label.value = (
            f"<b>{len(filtered)}</b> / {len(self.data[self._key()])} units"
        )
        self._updating = False
        self._update_display()

    def _update_display(self, _change=None):
        if self._updating or self._fig is None or self._ax is None:
            return
        unit_id = self.unit_slider.value
        self._ax.clear()
        if unit_id == "—":
            self._ax.set_visible(False)
            self.stats_html.value = "<i>No units match this filter.</i>"
            self._fig.canvas.draw_idle()
            return

        self._ax.set_visible(True)
        row = self.data[self._key()].set_index("unit_id").loc[unit_id]
        drift_status = "FAIL" if row.high_amplitude_drift else "PASS"
        dip_status = "FAIL" if row.non_unimodal else "PASS"
        self.stats_html.value = (
            f"<b>Unit {unit_id}</b><br>"
            f"Amplitude drift: {row.amplitude_drift:.3f} "
            f"(limit {self.amp_drift_threshold}) — {drift_status}<br>"
            f"Dip: {row.dip:.4f}; q = {row.dip_q:.4g}; n = {row.dip_n:,} "
            f"(alpha {self.dip_alpha}) — {dip_status}"
        )
        for i, chunk in enumerate(row.chunks):
            self._ax.hist(
                chunk,
                bins=self.n_bins,
                histtype="step",
                alpha=0.8,
                label=f"Time {i + 1}",
            )
        self._ax.set(title=f"Unit {unit_id}", xlabel="Spike amplitude", ylabel="Count")
        self._ax.legend(fontsize=8)
        self._fig.tight_layout()
        self._fig.canvas.draw_idle()

    def show(self):
        with plt.ioff():
            self._fig, self._ax = plt.subplots(figsize=(5, 3.5))
        controls = widgets.VBox(
            [self.session_dropdown, self.probe_dropdown, self.filter_radio]
        )
        details = widgets.VBox(
            [self.unit_slider, self.count_label, self.stats_html],
            layout=widgets.Layout(margin="0 0 0 20px"),
        )
        display(widgets.HBox([widgets.VBox([controls, self._fig.canvas]), details]))
        self._refresh_units()

Open the interactive unit-stability browser.

In [ ]:
browser = StabilityBrowser(
    probe_widget_data,
    amp_drift_threshold=amp_drift_threshold,
    dip_alpha=dip_alpha,
    n_bins=n_bins,
)
browser.show()